In [ ]:
# Instalacja pakietów
!pip install ultralytics

In [ ]:
# Załadowanie bibliotek
import os, cv2, numpy as np
import torch, torch.nn.functional as F
import matplotlib.pyplot as plt
from ultralytics import YOLO
from torchvision import transforms
from PIL import Image

In [ ]:
# Ustawienie katalogu roboczego
from google.colab import drive
drive.mount('/content/drive')
os.chdir('/content/drive/My Drive/Colab Notebooks/')

In [ ]:
# Załadowanie modelu
model = YOLO('yolov11n-cls.pt')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.model.to(device)

In [ ]:
# Wybór warstwy dla Grad-CAM (ostatnia warstwa konwolucyjna szkieletu)
target_layer = model.model.model[-2]
activations = []
gradients = []
def forward_hook(module, input, output):
    activations.append(output)
def backward_hook(module, grad_in, grad_out):
    gradients.append(grad_out[0])
target_layer.register_forward_hook(forward_hook)
target_layer.register_backward_hook(backward_hook)

In [ ]:
# Wstępne przetwarzanie obrazu
transform = transforms.Compose([
    transforms.Resize((150, 150)),
    transforms.ToTensor(),
])
def preprocess_image(img_path):
    img = Image.open(img_path).convert('RGB')
    tensor = transform(img).unsqueeze(0).to(device)
    return img, tensor

In [ ]:
# Funkcja Grad-CAM
def generate_gradcam(img_tensor, class_idx=None):
    activations.clear()
    gradients.clear()
    output = model.model(img_tensor)
    output_tensor = output[0] if isinstance(output, tuple) else output
    if class_idx is None:
        class_idx = output_tensor.argmax(dim = 1).item()
    loss = output_tensor[0, class_idx]
    model.model.zero_grad()
    loss.backward()
    grads = gradients[0]
    acts = activations[0]
    weights = torch.mean(grads, dim = (2, 3), keepdim=True)
    cam = torch.sum(weights * acts, dim = 1).squeeze()
    cam = F.relu(cam)
    cam = cam.detach().cpu().numpy()
    cam = (cam - cam.min()) / (cam.max() - cam.min())
    return cam, class_idx

In [ ]:
# Wizualizacja wyjaśnienia
def overlay_heatmap(original_img, cam, predicted_class):
    cam = cv2.resize(cam, original_img.size)
    heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
    heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
    original_np = np.array(original_img)
    overlay = cv2.addWeighted(original_np, 0.6, heatmap, 0.4, 0)
    plt.figure(figsize = (6, 6))
    plt.imshow(overlay)
    class_names = list(model.names.values())
    plt.title('Predykcja: ' + str(class_names[predicted_class]))
    plt.axis('off')
    plt.show()

In [ ]:
# Uruchomienie kodu
img_path = 'image.jpg' # Należy podać nazwę pliku obrazu
original_img, tensor = preprocess_image(img_path)
tensor.requires_grad_(True)
cam, predicted_class = generate_gradcam(tensor)
overlay_heatmap(original_img, cam, predicted_class)